#Two-Phase Training
### Phase 1 (epochs 1–10): Encoder fully frozen, decoder trains on cached features
### Phase 2 (epochs 11–15): ResNet layer4 unfrozen, fine-tuned with lr=1e-5

In [ ]:
!pip install -q torch torchvision tqdm kagglehub nltk

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
import random
import re
import os
import csv
from PIL import Image
from collections import Counter
from tqdm import tqdm
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_ROOT    = '/content/drive/MyDrive/FYP_Models/Two-Phase_Training'
os.makedirs(DRIVE_ROOT, exist_ok=True)

MODEL_PATH    = os.path.join(DRIVE_ROOT, 'resnet50_attention_model.pth')
VOCAB_PATH    = os.path.join(DRIVE_ROOT, 'vocab.pt')
FEATURE_PATH  = os.path.join(DRIVE_ROOT, 'resnet50_features.pt')

print('Drive directory ready:', DRIVE_ROOT)

Drive directory ready: /content/drive/MyDrive/FYP_Models/Two-Phase_Training


In [ ]:
import kagglehub
path = kagglehub.dataset_download('adityajn105/flickr8k')
IMAGE_DIR    = os.path.join(path, 'Images')
CAPTION_FILE = os.path.join(path, 'captions.txt')
print(len(os.listdir(IMAGE_DIR)), 'images found')
print('Caption file exists:', os.path.exists(CAPTION_FILE))

100%|██████████| 1.04G/1.04G [00:08<00:00, 137MB/s]

Extracting files...


8091 images found
Caption file exists: True


## Tokenizer & Vocabulary

In [ ]:
def tokenize(caption: str):
    """Lowercase, strip noise, keep <start>/<end> as single tokens."""
    caption = caption.lower().strip()
    caption = re.sub(r"[^a-z0-9<>' ]", ' ', caption)
    caption = caption.replace('<start>', ' <start> ').replace('<end>', ' <end> ')
    caption = re.sub(r'\s+', ' ', caption).strip()
    return caption.split()

# sanity check
print(tokenize('<start> A dog runs across the field . <end>'))

['<start>', 'a', 'dog', 'runs', 'across', 'the', 'field', '<end>']


In [ ]:
# ── Load captions ─────────────────────────────────────────────────────────────
captions = {}
with open(CAPTION_FILE, 'r') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 2:
            img_name     = row[0]
            caption_text = '<start> ' + row[1].strip() + ' <end>'
            captions.setdefault(img_name, []).append(caption_text)

print('Total images with captions:', len(captions))

Total images with captions: 8091


In [ ]:
# ── Build vocab ───────────────────────────────────────────────────────────────
word_counter = Counter()
SKIP_TOKENS  = {'<pad>', '<start>', '<end>', '<unk>'}
for caps_list in captions.values():
    for cap in caps_list:
        word_counter.update(t for t in tokenize(cap) if t not in SKIP_TOKENS)

SPECIALS = ['<pad>', '<start>', '<end>', '<unk>']
word2idx = {w: i for i, w in enumerate(SPECIALS)}
idx2word = {i: w for i, w in enumerate(SPECIALS)}

idx = len(SPECIALS)
MIN_FREQ = 5
for w, c in word_counter.items():
    if c >= MIN_FREQ and w not in word2idx:
        word2idx[w] = idx
        idx2word[idx] = w
        idx += 1

vocab_size = len(word2idx)
PAD_IDX    = word2idx['<pad>']
START_IDX  = word2idx['<start>']
END_IDX    = word2idx['<end>']
UNK_IDX    = word2idx['<unk>']

print(f'Vocabulary size: {vocab_size}')
print(f'  pad={PAD_IDX}, start={START_IDX}, end={END_IDX}, unk={UNK_IDX}')

torch.save((word2idx, idx2word), VOCAB_PATH)
print('Vocabulary saved.')

Vocabulary size: 2982
  pad=0, start=1, end=2, unk=3
Vocabulary saved.


In [ ]:
def caption_to_seq(caption: str):
    return [word2idx.get(w, UNK_IDX) for w in tokenize(caption)]

seq = caption_to_seq('<start> a dog runs <end>')
print('Example seq:', seq)
print('Decoded    :', [idx2word[i] for i in seq])

Example seq: [1, 4, 28, 123, 2]
Decoded    : ['<start>', 'a', 'dog', 'runs', '<end>']


## Transform, Encoder & Train/Val Split

In [ ]:
# ── Image transform (used everywhere) ────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── ResNet50 encoder — strip avgpool + fc to get (B, 2048, 7, 7) ──────────────
resnet  = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
encoder = nn.Sequential(*list(resnet.children())[:-2]).to(device)

# start fully frozen
for p in encoder.parameters():
    p.requires_grad = False
encoder.eval()

print('Encoder ready — fully frozen for Phase 1.')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


Encoder ready — fully frozen for Phase 1.


In [ ]:
# ── Train / val split ─────────────────────────────────────────────────────────
all_images = list(captions.keys())
random.seed(42)
random.shuffle(all_images)
split        = int(0.9 * len(all_images))
train_images = all_images[:split]
val_images   = all_images[split:]
print(f'Train: {len(train_images)} | Val: {len(val_images)}')

Train: 7281 | Val: 810


## Feature Cache (Phase 1)
Pre-extract and cache ResNet features. Phase 1 training uses these cached features
so it's fast. Phase 2 runs the encoder live since layer4 weights are changing.

In [ ]:
if os.path.exists(FEATURE_PATH):
    print('Loading cached features from Drive...')
    all_features = torch.load(FEATURE_PATH, map_location='cpu', weights_only=False)
    print(f'Loaded features for {len(all_features)} images.')
else:
    print('Extracting features — this takes a few minutes...')
    all_features = {}
    with torch.no_grad():
        for img_name in tqdm(all_images):
            img_path = os.path.join(IMAGE_DIR, img_name)
            try:
                img   = Image.open(img_path).convert('RGB')
                img_t = transform(img).unsqueeze(0).to(device)
                feat  = encoder(img_t)                          # (1,2048,7,7)
                feat  = feat.squeeze(0).permute(1,2,0).reshape(49,2048).cpu()
                all_features[img_name] = feat
            except Exception as e:
                print(f'  Skipping {img_name}: {e}')
    torch.save(all_features, FEATURE_PATH)
    print(f'Saved features for {len(all_features)} images.')

Extracting features — this takes a few minutes...


100%|██████████| 8091/8091 [01:46<00:00, 76.18it/s]


Saved features for 8091 images.


## Datasets & DataLoaders
Two dataset classes:
- `CachedDataset` — returns pre-extracted features (fast, used in Phase 1)
- `LiveDataset` — returns raw images, encoder runs each batch (Phase 2)

In [ ]:
from torch.utils.data import Dataset, DataLoader


def collate_cached(batch):
    """Collate for CachedDataset — features already extracted."""
    feats, seqs = zip(*batch)
    feats  = torch.stack(feats)
    max_len = max(s.size(0) for s in seqs)
    padded  = torch.full((len(seqs), max_len), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :s.size(0)] = s
    return feats, padded


def collate_live(batch):
    """Collate for LiveDataset — returns raw image tensors."""
    imgs, seqs = zip(*batch)
    imgs   = torch.stack(imgs)
    max_len = max(s.size(0) for s in seqs)
    padded  = torch.full((len(seqs), max_len), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :s.size(0)] = s
    return imgs, padded


class CachedDataset(Dataset):
    """Uses pre-extracted features — fast. For Phase 1."""
    def __init__(self, image_names, max_len=50):
        self.data = []
        for img_name in image_names:
            if img_name not in all_features:
                continue
            for cap in captions[img_name]:
                seq = caption_to_seq(cap)
                if len(seq) <= max_len:
                    self.data.append((img_name, seq))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_name, seq = self.data[idx]
        return all_features[img_name], torch.tensor(seq, dtype=torch.long)


class LiveDataset(Dataset):
    """Loads raw images — encoder runs live each batch. For Phase 2."""
    def __init__(self, image_names, max_len=50):
        self.data = []
        for img_name in image_names:
            img_path = os.path.join(IMAGE_DIR, img_name)
            if not os.path.exists(img_path):
                continue
            for cap in captions[img_name]:
                seq = caption_to_seq(cap)
                if len(seq) <= max_len:
                    self.data.append((img_path, seq))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_path, seq = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        return transform(img), torch.tensor(seq, dtype=torch.long)


# Phase 1 loaders (batch=64, cached)
P1_BATCH = 64
p1_train_loader = DataLoader(CachedDataset(train_images), batch_size=P1_BATCH,
                             shuffle=True,  collate_fn=collate_cached, num_workers=2)
p1_val_loader   = DataLoader(CachedDataset(val_images),   batch_size=P1_BATCH,
                             shuffle=False, collate_fn=collate_cached, num_workers=2)

# Phase 2 loaders (batch=32, live — smaller batch due to higher GPU memory)
P2_BATCH = 32
p2_train_loader = DataLoader(LiveDataset(train_images), batch_size=P2_BATCH,
                             shuffle=True,  collate_fn=collate_live, num_workers=2)
p2_val_loader   = DataLoader(LiveDataset(val_images),   batch_size=P2_BATCH,
                             shuffle=False, collate_fn=collate_live, num_workers=2)

print(f'Phase 1 — Train: {len(p1_train_loader.dataset)} | Val: {len(p1_val_loader.dataset)}')
print(f'Phase 2 — Train: {len(p2_train_loader.dataset)} | Val: {len(p2_val_loader.dataset)}')

Phase 1 — Train: 36405 | Val: 4050
Phase 2 — Train: 36405 | Val: 4050


## Model

In [ ]:
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attn_dim):
        super().__init__()
        self.W_enc = nn.Linear(encoder_dim, attn_dim)
        self.W_dec = nn.Linear(decoder_dim, attn_dim)
        self.V     = nn.Linear(attn_dim, 1)

    def forward(self, encoder_out, h):
        e       = self.W_enc(encoder_out)             # (B, 49, attn_dim)
        d       = self.W_dec(h).unsqueeze(1)          # (B, 1,  attn_dim)
        score   = self.V(torch.tanh(e + d))           # (B, 49, 1)
        alpha   = torch.softmax(score, dim=1)         # (B, 49, 1)
        context = (alpha * encoder_out).sum(1)        # (B, encoder_dim)
        return context, alpha.squeeze(-1)


class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, encoder_dim=2048,
                 decoder_dim=512, attn_dim=256, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention = Attention(encoder_dim, decoder_dim, attn_dim)
        self.lstm      = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)
        self.init_c    = nn.Linear(encoder_dim, decoder_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(decoder_dim, vocab_size)

    def _init_hidden(self, encoder_out):
        mean_enc = encoder_out.mean(dim=1)
        return torch.tanh(self.init_h(mean_enc)), torch.tanh(self.init_c(mean_enc))

    def forward(self, encoder_out, captions, teacher_forcing_ratio=1.0):
        B, T    = captions.size()
        h, c    = self._init_hidden(encoder_out)
        outputs = []
        input_token = captions[:, 0]

        for t in range(T - 1):
            emb            = self.embedding(input_token)
            context, _     = self.attention(encoder_out, h)
            h, c           = self.lstm(torch.cat([emb, context], dim=1), (h, c))
            logits         = self.fc(self.dropout(h))
            outputs.append(logits)
            use_gt         = random.random() < teacher_forcing_ratio
            input_token    = captions[:, t+1] if use_gt else logits.argmax(1)

        return torch.stack(outputs, dim=1)  # (B, T-1, V)


# ── Always re-run this cell to freshly initialise the model ───────────────────
model = DecoderWithAttention(
    vocab_size  = vocab_size,
    embed_dim   = 256,
    encoder_dim = 2048,
    decoder_dim = 512,
    attn_dim    = 256,
    dropout     = 0.5,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Decoder parameters: {total_params:,}')
print('Model freshly initialised.')

Decoder parameters: 10,818,727
Model freshly initialised.


## Helper: encode a batch of raw images
Used only in Phase 2 where the encoder is live.

In [ ]:
def encode_images(imgs):
    """Run encoder on a batch of raw image tensors. Returns (B, 49, 2048)."""
    imgs = imgs.to(device)
    feat = encoder(imgs)                         # (B, 2048, 7, 7)
    feat = feat.permute(0, 2, 3, 1)              # (B, 7, 7, 2048)
    feat = feat.reshape(feat.size(0), 49, 2048)  # (B, 49, 2048)
    return feat

print('encode_images() helper ready.')

encode_images() helper ready.


## Phase 1 — Frozen Encoder (10 epochs)
Fast training using cached features. Decoder learns from scratch.

In [ ]:
criterion  = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.05)

# Phase 1 optimizer — decoder only
p1_optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
P1_EPOCHS    = 10
p1_scheduler = optim.lr_scheduler.CosineAnnealingLR(p1_optimizer, T_max=P1_EPOCHS, eta_min=1e-5)

print('Phase 1 optimizer ready.')

Phase 1 optimizer ready.


In [ ]:
def run_epoch_cached(loader, optimizer, tf_ratio, train=True):
    """One epoch using pre-cached features."""
    model.train() if train else model.eval()
    total_loss = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for feats, caps in loader:
            feats = feats.to(device)
            caps  = caps.to(device)
            outputs = model(feats, caps, teacher_forcing_ratio=tf_ratio)
            loss    = criterion(outputs.reshape(-1, vocab_size), caps[:, 1:].reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)


print('Phase 1 training starting...')
best_val_loss = float('inf')

for epoch in range(1, P1_EPOCHS + 1):
    tf_ratio   = max(0.75, 1.0 - 0.005 * (epoch - 1))
    train_loss = run_epoch_cached(p1_train_loader, p1_optimizer, tf_ratio, train=True)
    val_loss   = run_epoch_cached(p1_val_loader,   p1_optimizer, 1.0,      train=False)
    p1_scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch':    epoch,
            'phase':    1,
            'model':    model.state_dict(),
            'word2idx': word2idx,
            'idx2word': idx2word,
        }, MODEL_PATH)
        flag = '  ✓ saved'
    else:
        flag = ''

    print(f'[P1] Epoch {epoch:2d}/{P1_EPOCHS} | Train {train_loss:.4f} | Val {val_loss:.4f} | TF {tf_ratio:.2f}{flag}')

print('\nPhase 1 complete.')

Phase 1 training starting...
[P1] Epoch  1/10 | Train 4.0095 | Val 3.4740 | TF 1.00  ✓ saved
[P1] Epoch  2/10 | Train 3.3207 | Val 3.2831 | TF 0.99  ✓ saved
[P1] Epoch  3/10 | Train 3.0770 | Val 3.2126 | TF 0.99  ✓ saved
[P1] Epoch  4/10 | Train 2.8998 | Val 3.1882 | TF 0.98  ✓ saved
[P1] Epoch  5/10 | Train 2.7448 | Val 3.1748 | TF 0.98  ✓ saved
[P1] Epoch  6/10 | Train 2.6184 | Val 3.1832 | TF 0.97
[P1] Epoch  7/10 | Train 2.5118 | Val 3.1893 | TF 0.97
[P1] Epoch  8/10 | Train 2.4203 | Val 3.2004 | TF 0.96
[P1] Epoch  9/10 | Train 2.3675 | Val 3.2118 | TF 0.96
[P1] Epoch 10/10 | Train 2.3355 | Val 3.2161 | TF 0.95

Phase 1 complete.


## Phase 2 — Unfreeze ResNet layer4 (5 epochs)
Load the best Phase 1 checkpoint, unfreeze only `layer4` of ResNet,
and fine-tune with a very small LR (`1e-5`) to avoid destroying ImageNet weights.
The decoder continues training at its existing LR (`1e-4`).
Batch size is halved to 32 to handle the extra GPU memory from encoder gradients.

In [ ]:
# ── Load best Phase 1 checkpoint ─────────────────────────────────────────────
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
print(f'Loaded Phase 1 best model from epoch {ckpt["epoch"]}.')

# ── Unfreeze layer4 only ──────────────────────────────────────────────────────
# layer4 is the 7th child of ResNet (index 6 in the Sequential encoder)
# children: 0=conv1 1=bn1 2=relu 3=maxpool 4=layer1 5=layer2 6=layer3 7=layer4
encoder.train()  # switch encoder to train mode for Phase 2
for p in encoder.parameters():
    p.requires_grad = False   # keep everything frozen first

layer4 = list(encoder.children())[7]  # layer4
for p in layer4.parameters():
    p.requires_grad = True    # unfreeze layer4 only

unfrozen = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f'Unfrozen encoder parameters (layer4 only): {unfrozen:,}')

# ── Phase 2 optimizer — two param groups with different LRs ──────────────────
# encoder layer4: very small LR so we don't destroy ImageNet weights
# decoder:        smaller than Phase 1 since it's already partially trained
p2_optimizer = optim.AdamW([
    {'params': filter(lambda p: p.requires_grad, encoder.parameters()), 'lr': 1e-5},
    {'params': model.parameters(),                                       'lr': 1e-4},
], weight_decay=5e-4)

P2_EPOCHS    = 5
p2_scheduler = optim.lr_scheduler.CosineAnnealingLR(p2_optimizer, T_max=P2_EPOCHS, eta_min=1e-6)

print('Phase 2 optimizer ready.')

Loaded Phase 1 best model from epoch 5.
Unfrozen encoder parameters (layer4 only): 14,964,736
Phase 2 optimizer ready.


In [ ]:
def run_epoch_live(loader, optimizer, tf_ratio, train=True):
    """One epoch where the encoder runs live (no cached features)."""
    model.train() if train else model.eval()
    encoder.train() if train else encoder.eval()
    total_loss = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, caps in tqdm(loader, desc='  batch', leave=False):
            caps  = caps.to(device)
            feats = encode_images(imgs)  # (B, 49, 2048)
            outputs = model(feats, caps, teacher_forcing_ratio=tf_ratio)
            loss    = criterion(outputs.reshape(-1, vocab_size), caps[:, 1:].reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),   5.0)
                nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)  # tighter clip for encoder
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)


print('Phase 2 training starting (this will be slower — encoder runs live)...')

for epoch in range(1, P2_EPOCHS + 1):
    tf_ratio   = 0.75  # keep fixed during fine-tuning
    train_loss = run_epoch_live(p2_train_loader, p2_optimizer, tf_ratio, train=True)
    val_loss   = run_epoch_live(p2_val_loader,   p2_optimizer, 1.0,      train=False)
    p2_scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch':    P1_EPOCHS + epoch,
            'phase':    2,
            'model':    model.state_dict(),
            'encoder':  encoder.state_dict(),  # save encoder too since layer4 changed
            'word2idx': word2idx,
            'idx2word': idx2word,
        }, MODEL_PATH)
        flag = '  ✓ saved'
    else:
        flag = ''

    print(f'[P2] Epoch {epoch:2d}/{P2_EPOCHS} | Train {train_loss:.4f} | Val {val_loss:.4f}{flag}')

print('\nPhase 2 complete.')

Phase 2 training starting (this will be slower — encoder runs live)...


[P2] Epoch  1/5 | Train 3.2894 | Val 3.2031


[P2] Epoch  2/5 | Train 3.1319 | Val 3.2015


[P2] Epoch  3/5 | Train 3.0736 | Val 3.2006


[P2] Epoch  4/5 | Train 3.0212 | Val 3.2019


[P2] Epoch  5/5 | Train 3.0072 | Val 3.2023

Phase 2 complete.


## Beam Search

In [ ]:
@torch.no_grad()
def beam_search(feature, beam_size=5, max_len=50):
    """
    feature : (49, 2048) CPU tensor.
    Returns : list of token strings (no <start>/<end>).
    """
    model.eval()
    enc  = feature.unsqueeze(0).to(device)        # (1, 49, 2048)
    h, c = model._init_hidden(enc)

    beams     = [([START_IDX], 0.0, h.clone(), c.clone())]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for seq, score, h, c in beams:
            tok     = torch.tensor([seq[-1]], device=device)
            emb     = model.embedding(tok)
            ctx, _  = model.attention(enc, h)
            h_new, c_new = model.lstm(torch.cat([emb, ctx], dim=1), (h, c))
            logits  = model.fc(model.dropout(h_new))
            logits[0, PAD_IDX] = -1e9
            log_probs = torch.log_softmax(logits, dim=-1)
            topk_lp, topk_idx = log_probs[0].topk(beam_size)

            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                new_seq   = seq + [idx]
                new_score = score + lp
                if idx == END_IDX:
                    completed.append((new_seq, new_score))
                else:
                    new_beams.append((new_seq, new_score, h_new, c_new))

        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_size]
        if not beams:
            break

    if not completed:
        completed = [(b[0], b[1]) for b in beams]

    best_seq, _ = max(completed, key=lambda x: x[1] / len(x[0]))
    return [idx2word[i] for i in best_seq if i not in (START_IDX, END_IDX, PAD_IDX)]


print('Beam search ready.')

Beam search ready.


## BLEU-4 Evaluation

In [ ]:
# ── Load best checkpoint (works for both Phase 1 and Phase 2 checkpoints) ─────
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])

# if the best checkpoint is from Phase 2, restore encoder layer4 weights too
if ckpt.get('phase', 1) == 2 and 'encoder' in ckpt:
    encoder.load_state_dict(ckpt['encoder'])
    print(f'Loaded Phase 2 checkpoint from epoch {ckpt["epoch"]} (encoder layer4 restored).')
else:
    print(f'Loaded Phase 1 checkpoint from epoch {ckpt["epoch"]}.')

encoder.eval()
model.eval()

# ── Extract features fresh for val set using current encoder state ─────────────
print('Extracting val features with current encoder...')
val_features = {}
with torch.no_grad():
    for img_name in tqdm(val_images):
        img_path = os.path.join(IMAGE_DIR, img_name)
        if not os.path.exists(img_path):
            continue
        img   = Image.open(img_path).convert('RGB')
        img_t = transform(img).unsqueeze(0).to(device)
        feat  = encoder(img_t).squeeze(0).permute(1,2,0).reshape(49,2048).cpu()
        val_features[img_name] = feat

# ── BLEU-4 ────────────────────────────────────────────────────────────────────
smoothie = SmoothingFunction().method4
refs_all, hyps_all = [], []

for img_name in tqdm(val_images, desc='BLEU eval'):
    if img_name not in val_features:
        continue
    hyp  = beam_search(val_features[img_name], beam_size=5)
    refs = [tokenize(c) for c in captions[img_name]]
    hyps_all.append(hyp)
    refs_all.append(refs)

bleu4 = corpus_bleu(refs_all, hyps_all, smoothing_function=smoothie)
print(f'\nValidation BLEU-4: {bleu4:.4f}')

Loaded Phase 1 checkpoint from epoch 5.
Extracting val features with current encoder...


BLEU eval: 100%|██████████| 810/810 [02:33<00:00,  5.26it/s]



Validation BLEU-4: 0.2173


## Generate a Caption for Your Own Image

In [ ]:
# ── Always load best checkpoint before generating ─────────────────────────────
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
if ckpt.get('phase', 1) == 2 and 'encoder' in ckpt:
    encoder.load_state_dict(ckpt['encoder'])
    print(f'Loaded Phase 2 model from epoch {ckpt["epoch"]}.')
else:
    print(f'Loaded Phase 1 model from epoch {ckpt["epoch"]}.')
encoder.eval()
model.eval()


@torch.no_grad()
def generate_caption(image_path: str, beam_size: int = 5):
    img   = Image.open(image_path).convert('RGB')
    img_t = transform(img).unsqueeze(0).to(device)
    feat  = encoder(img_t).squeeze(0).permute(1,2,0).reshape(49,2048).cpu()
    return ' '.join(beam_search(feat, beam_size=beam_size))


from google.colab import files
uploaded = files.upload()
for image_path in uploaded.keys():
    caption = generate_caption(image_path)
    print(f'\n🖼️  {image_path}')
    print(f'Caption: {caption}')